# 01 Exploratory Data Analysis (EDA)
**Project Title**: Intelligent Cloud Server Anomaly Detection and Real-Time Monitoring System

This notebook performs exploratory analysis on our validated server metrics dataset. We examine distributions, temporal behaviors, server-level profiles, and correlations to establish baseline server characteristics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set plot styles
sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 10, 'axes.labelsize': 11, 'axes.titlesize': 12})

# Load the validated dataset
data_path = "../data/processed/validated_historical_metrics.csv"
df = pd.read_csv(data_path)
df["timestamp"] = pd.to_datetime(df["timestamp"])

print(f"Dataset Shape: {df.shape}")
df.head()

## 1. Summary Statistics
Let's inspect numerical descriptions of the server metrics.

In [ ]:
df.describe().T

## 2. Server-Wise Observations Counts

In [ ]:
df["server_id"].value_counts()

## 3. Metric Distribution Profiles
We plot histogram distributions with Kernel Density Estimates (KDE) to inspect shapes, skewness, and extreme value patterns.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 11))
metrics = ["cpu_usage", "memory_usage", "disk_usage", "network_traffic", "response_time", "request_rate"]
titles = [
    "CPU Utilization (%)", "Memory Utilization (%)", "Disk Write Throughput (MB/s)", 
    "Network Traffic (MB/s)", "Response Time (ms)", "Request Rate (req/sec)"
]
colors = ["#4A90E2", "#50E3C2", "#F5A623", "#D0021B", "#9013FE", "#B8E986"]

for idx, (metric, title, color) in enumerate(zip(metrics, titles, colors)):
    ax = axes[idx // 2, idx % 2]
    sns.histplot(df[metric], kde=True, ax=ax, color=color, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Count")
    
plt.tight_layout()
plt.show()

## 4. Server Workload Comparison
We use boxplots to compare the load profiles across `SERVER-01`, `SERVER-02`, and `SERVER-03`.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(14, 11))
for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[idx // 2, idx % 2]
    sns.boxplot(x="server_id", y=metric, data=df, ax=ax, hue="server_id", legend=False, palette="Set2")
    ax.set_title(f"Comparison: {title}")
    ax.set_xlabel("")
    ax.set_ylabel("")
    
plt.tight_layout()
plt.show()

## 5. Metric Correlation Heatmap
Examining linear dependencies between resource metrics.

In [ ]:
plt.figure(figsize=(8, 6))
corr = df[metrics].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title("Metric Correlation Heatmap")
plt.tight_layout()
plt.show()

## 6. Time Series Workload Profiles (3-Day Sample)
Let's trace resource metrics over time for a 3-day window to look for peaks, cyclical workloads, and correlations.

In [ ]:
sample_df = df[df["timestamp"] <= df["timestamp"].min() + pd.Timedelta(days=3)]
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

sns.lineplot(data=sample_df, x="timestamp", y="cpu_usage", hue="server_id", ax=axes[0], alpha=0.85)
axes[0].set_title("CPU Utilization Profile (%) - 3-Day Sample")
axes[0].set_ylabel("CPU %")
axes[0].get_legend().remove()

sns.lineplot(data=sample_df, x="timestamp", y="network_traffic", hue="server_id", ax=axes[1], alpha=0.85)
axes[1].set_title("Network Traffic Profile (MB/s) - 3-Day Sample")
axes[1].set_ylabel("Traffic (MB/s)")
axes[1].get_legend().remove()

sns.lineplot(data=sample_df, x="timestamp", y="response_time", hue="server_id", ax=axes[2], alpha=0.85)
axes[2].set_title("Response Time Profile (ms) - 3-Day Sample")
axes[2].set_ylabel("RT (ms)")

plt.xlabel("Time")
plt.tight_layout()
plt.show()